# 03a - RM-a: full fine-tuning IndoBERT

Baseline penelitian. Seluruh 109 juta parameter IndoBERT diperbarui, sehingga
skenario ini yang paling mahal sekaligus menjadi pembanding performa untuk dua
strategi ringan.

Notebook ini menjalankan SATU konfigurasi: baseline kanonik IndoNLU/Wilie (2020)
`lr=2e-5, epochs=5, batch=16, warmup=0,1, wd=0,01`. Eksplorasi hyperparameter
ada di `04_tuning_campaign.ipynb`.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [1]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "baseline"

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])

c:\Penelitian\IndoBERT-with-RAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-02 14:30:24,277 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


device        : cuda
encoder       : indobenchmark/indobert-base-p2
keluaran      : C:\Penelitian\IndoBERT-with-RAC\outputs\baseline
train/val/test: [6588, 1402, 1405]


## 1. Catat konteks hardware

In [2]:
hardware = runner.write_hardware()
for key, value in hardware.items():
    print(f"  {key:16s}: {value}")

  gpu             : NVIDIA GeForce RTX 3050 Laptop GPU
  cuda_available  : True
  cuda_version    : 13.0
  torch           : 2.12.1+cu130
  transformers    : 5.12.1
  vram_total_mb   : 4096
  recorded_at     : 2026-09-02 14:30:28
  nvidia_smi      : NVIDIA GeForce RTX 3050 Laptop GPU, 592.00, 4096 MiB


Angka efisiensi hanya bisa ditafsirkan bersama konteks ini, dan hanya sebanding
bila seluruh skenario diukur pada hardware dan sesi yang sama.

## 2. Konfigurasi

In [3]:
from src.models.schemas import RMAConfig

config = RMAConfig()
print(config.model_dump())
print(f"\nbatch efektif {config.batch} dicapai lewat micro-batch "
      f"{config.effective_micro_batch} x akumulasi {config.grad_accum}")

{'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 8, 'seed': 42}

batch efektif 16 dicapai lewat micro-batch 8 x akumulasi 2


Akumulasi gradien membuat RUMUS gradiennya ekuivalen dengan batch besar (BERT
memakai LayerNorm, bukan BatchNorm, dan loss dibagi jumlah akumulasi), tetapi
RUN-nya tidak identik: `DataLoader` dengan ukuran batch berbeda mengonsumsi RNG
secara berbeda, sehingga mask dropout dan komposisi tiap batch ikut berubah.

Terukur di kampanye ini: `micro_batch` efektif 8 versus 16 pada konfigurasi yang
sama persis memberi val F1-macro 0,974873 versus 0,977266. Karena itu
`micro_batch` harus dikunci untuk seluruh sel satu grid dan disebutkan di Bab 4
sebagai bagian konfigurasi, bukan diperlakukan sebagai knob memori bebas.

## 3. Jalankan

In [4]:
row = runner.run(
    "rma",
    config.model_dump(),
    note="baseline kanonik IndoNLU/Wilie 2020, titik acuan seluruh grid",
)

print(f"run #{row['run_id']}")
print(f"  val F1-macro    : {row['val_f1_macro']:.4f}")
print(f"  val F1 judi     : {row['val_f1_judi']:.4f}")
print(f"  epoch terbaik   : {row['best_epoch']} dari {row['epochs']}")
print(f"  waktu latih     : {row['train_time_s']:.1f} s")
print(f"  peak GPU memory : {row['peak_mem_mb']:.0f} MB")
print(f"  trainable params: {row['trainable_params']:,}")

2026-09-02 14:30:29,087 | INFO     | src.services.campaign | [rma] RUN #1 {'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 8, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6656.51it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-02 14:30:34,360 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-02 14:33:41,303 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9398
2026-09-02 14:36:33,359 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9647
2026-09-02 14:39:17,580 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9749
2026-09-02 14:42:02,323 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9740
2026-09-02 14:44:47,701 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9736
2026-09-02 14:44:48,273 | INFO     | src.services.run_log | Juara baru untuk rma: val F1-macro 0.9749 (sebelumnya -1.0000)
2026-09-02 14:45:01,535 | INFO     | src.services.campaign | [rma] RUN #1 val F1-macro 0.9749 (epoch terbaik 3)
run #1
  val F1-macro    : 0.9749
  val F1 judi     : 0.9589
  epoch terbaik   : 3 dari 5
  waktu latih     : 852.9 s
  peak GPU memory : 2339 MB
  trainable params: 109,485,314

## 4. Kurva per epoch

In [5]:
import pandas as pd

history = pd.read_csv(OUT_DIR / "history" / "rma_history.csv")
history[history["run_id"] == row["run_id"]]

,run_id,epoch,train_loss,val_f1_macro,val_acc,val_f1_judi,val_precision_judi,val_recall_judi
0,1,1,0.216257,0.939813,0.962197,0.903108,0.851724,0.961089
1,1,2,0.079411,0.964691,0.978602,0.942529,0.928302,0.957198
2,1,3,0.039179,0.974873,0.985021,0.958904,0.964567,0.953307
3,1,4,0.015346,0.973953,0.984308,0.957529,0.950192,0.964981
4,1,5,0.010588,0.973636,0.984308,0.956863,0.964427,0.949416


`overfit_signal` di baris riwayat bernilai True bila epoch terbaik bukan epoch
terakhir, yaitu tanda bahwa menambah epoch justru memperburuk validasi.

Figur kurva tersimpan di `outputs/baseline/figures/rma_run{id}_curve.png`.

## Ringkasan

Angka di atas adalah baseline satu konfigurasi, bukan hasil final. Konfigurasi
final ditentukan lewat kampanye di `04_tuning_campaign.ipynb`, dan angka test
baru dibuka sekali di `05_final_benchmark.ipynb`.

Lanjut ke `03b_rmb_frozen.ipynb`.